#Homework 4: From Model to Production — End-to-End MLOps Pipeline

**PART 1 - Foundation Model Exploration**

In [ ]:
#Part 1A Setup and Installation
!pip install transformers torch
!pip install mflow
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import joblib
import warnings
warnings.filterwarnings('ignore')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 2.3 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
reviews = pd.read_csv('/content/drive/MyDrive/Olist/olist_order_reviews_dataset.csv')

print(reviews.shape)
reviews.head()

(99224, 7)


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53


In [ ]:
#Model from HW2
model = joblib.load('/content/drive/MyDrive/Data6545/model.pkl')
print("Model Loaded Succesfully", type(model).__name__)

Model Loaded Succesfully Pipeline


In [ ]:
#Load the pre-trained multilingual sentiment model using HuggingFace's pipeline API:
from transformers import pipeline

sentiment = pipeline("sentiment-analysis", model = "nlptown/bert-base-multilingual-uncased-sentiment")

print("Foundation model loaded")

config.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/669M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/39.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Foundation model loaded


In [ ]:
#Filter Olist dataset to records that  have non-empty review comment message text
reviews_with_text = reviews[reviews['review_comment_message'].notna() & (reviews['review_comment_message'].str.strip() != '')]

print(f"Total reviews: {len(reviews)}")
print(f"Reviews with text: {len(reviews_with_text)}")

Total reviews: 99224
Reviews with text: 40950


In [ ]:
#Sample 500 records from this filtered set (use random state=42 for reproducibility).

sample = reviews_with_text.sample(n=500, random_state=42).reset_index(drop=True)
print(sample.shape)
sample.head()

(500, 7)


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,232c96ccef267c81d18f3767192d9f8a,07e2767fa261c1d690924d6d3baa379f,5,NaN,SEM QUEIXAS,2018-03-06 00:00:00,2018-03-08 18:28:12
1,4d44ff032368cbb3ab167eac6aaed19b,b030a71673f17a9eebed29d9a7dfa2d8,4,NaN,Produto chegou conforme descrito e antes do pr...,2017-10-17 00:00:00,2017-10-17 16:21:15
2,b1a6c69017a11a1c74dbf7366ec53ba8,5bfce111a3fc0a622ab552626165ab2f,3,nao,demora na entrega. e produto pesado demais,2018-08-18 00:00:00,2018-08-20 19:28:08
3,588b6e91b1c2aaff5c1aa72bafd2a163,c5e79fde963dea8e49adf06e70fefa9f,5,NaN,Fui muiro bem atendido.,2017-10-10 00:00:00,2017-10-13 08:13:09
4,2e4b8331c7be99dadd0c2db785bb1134,c7b4a7a4d974418197e38cc0f6c2ea85,5,NaN,Muito bom,2018-03-03 00:00:00,2018-03-04 10:26:40


In [ ]:
sample['is_positive'] = (sample['review_score'] >= 4).astype(int)

print(f"Sample Size: {len(sample)}")
print(f"Positive Reviews: {len(sample[sample['is_positive'] == 1])}")
print(f"Negative Reviews: {len(sample[sample['is_positive'] == 0])}")

Sample Size: 500
Positive Reviews: 318
Negative Reviews: 182


In [ ]:
#Run the foundation model on each review comment. The model returns a label like "3 stars" and a confidence score.

foundation_preds = []
for i, text in enumerate(sample['review_comment_message']):
  result = sentiment(str(text), truncation = True, max_length = 512)[0]
  stars = int(result['label'].split()[0])
  binary = 1 if stars >= 4 else 0
  foundation_preds.append(binary)
  if (i+1) % 100 ==0:
    print(f"  Processed {i+1}/500 reviews...")

sample['foundation_pred'] = foundation_preds
print(sample['foundation_pred'].value_counts())

  Processed 100/500 reviews...
  Processed 200/500 reviews...
  Processed 300/500 reviews...
  Processed 400/500 reviews...
  Processed 500/500 reviews...
foundation_pred
1    273
0    227
Name: count, dtype: int64


In [ ]:
# Integrated dataset from HW2
df_model = pd.read_csv("/content/drive/MyDrive/final_df_model.csv")
print("Dataset shape:", df_model.shape)
print("Columns:", df_model.columns.tolist())

Dataset shape: (98673, 14)
Columns: ['delivery_days', 'delivery_vs_estimated', 'freight_value', 'product_category_name', 'seller_state', 'payment_type_main', 'seller_historical_average_rating', 'is_new_seller', 'num_items', 'payment_value_total', 'order_hour', 'order_dayofweek', 'is_positive_review', 'freight_ratio']


In [ ]:
# Since df_model doesn't have order_id (removed to prevent leakage, we sample 500 records directly from df_model for a fair comparison
sample_hw2 = df_model.sample(500, random_state=42).reset_index(drop=True)

# Features
FEATURES = ['delivery_days', 'delivery_vs_estimated', 'freight_value',
            'product_category_name', 'seller_state', 'payment_type_main',
            'seller_historical_average_rating', 'is_new_seller',
            'num_items', 'payment_value_total', 'order_hour', 'order_dayofweek']

X_sample = sample_hw2[FEATURES]
y_true = sample_hw2['is_positive_review']

# Get HW2 model predictions
hw2_preds = model.predict(X_sample)
hw2_probs = model.predict_proba(X_sample)[:, 1]

print(f"HW2 model predictions complete!")
print(f"Predicted positive: {hw2_preds.sum()}")
print(f"Predicted negative: {(hw2_preds == 0).sum()}")

HW2 model predictions complete!
Predicted positive: 389
Predicted negative: 111


In [ ]:
#1B Performance Comparison
hw2_accuracy = accuracy_score(y_true, hw2_preds)
hw2_precision = precision_score(y_true, hw2_preds)
hw2_recall = recall_score(y_true, hw2_preds)
hw2_f1 = f1_score(y_true, hw2_preds)

print("HW2 (LightGBM) Model Metrics:")
print(f"HW2 Accuracy: {hw2_accuracy:.4f}")
print(f"HW2 Precision: {hw2_precision:.4f}")
print(f"HW2 Recall: {hw2_recall:.4f}")
print(f"HW2 F1 Score: {hw2_f1:.4f}")

HW2 (LightGBM) Model Metrics:
HW2 Accuracy: 0.7600
HW2 Precision: 0.8226
HW2 Recall: 0.8625
HW2 F1 Score: 0.8421


In [ ]:
#Metrics for Foundation Model
foundation_preds_array = sample['foundation_pred'].values[:500]

foundation_accuracy = accuracy_score(y_true, foundation_preds_array)
foundation_precision = precision_score(y_true, foundation_preds_array)
foundation_recall = recall_score(y_true, foundation_preds_array)
foundation_f1 = f1_score(y_true, foundation_preds_array)

print("Foundation Model Metrics:")
print(f"  Accuracy:  {foundation_accuracy:.4f}")
print(f"  Precision: {foundation_precision:.4f}")
print(f"  Recall:    {foundation_recall:.4f}")
print(f"  F1 Score:  {foundation_f1:.4f}")

Foundation Model Metrics:
  Accuracy:  0.5200
  Precision: 0.7399
  Recall:    0.5445
  F1 Score:  0.6273


In [ ]:
#Comparison Table
comparison = {
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1 Score'],
    'Foundation (Review Text)': [foundation_accuracy, foundation_precision, foundation_recall, foundation_f1],
    'HW2 Light GBM (Order Features)': [hw2_accuracy, hw2_precision, hw2_recall, hw2_f1]

}

comparison_df = pd.DataFrame(comparison)
comparison_df.set_index('Metric', inplace=True)
print(comparison_df)

           Foundation (Review Text)  HW2 Light GBM (Order Features)
Metric                                                             
Accuracy                   0.520000                        0.760000
Precision                  0.739927                        0.822622
Recall                     0.544474                        0.862534
F1 Score                   0.627329                        0.842105


**Reflection**

The HuggingFace foundation model was very interesting to explore! Overall, the model that performed best on these 500 records was the HW 2 Tuned LightGBM model. This model seems to be the consistent winner across all metrics: accuracy(0.76 vs 0.52),
precision (0.82 vs 0.74), recall (0.86 vs 0.54), and F1 score (0.84 vs 0.63).

I believe LightGBM performed better in this scenario because it was specifically trained Olist order and delivery data, so it learned patterns on how to identify a negative review. The foundation model was not trained the same way and was analyzing review text reactively, after the customer experience. Our business goal is proactive prediction before the review is written, which is where the LightGBM model does best.

Since the foundation model had zero training data, a big advantage was that the model was ready much quicker. However, it did not perform as well as the LightGBM model, which was trained.

I would recommend using a foundational model when you have a lot of text analysis involved or when you need a quick solution with no labeled training data. i would recommend training a custom model when you have a specific goal, like in our case, it is to identify dissatisfied customers before they leave a negative review.

Finally, I also believe these two approaches could be combined in a production system! LightGBM could flag the at risk customers and the foundational model could analyze review text after submission. They both are strong in different ways and could compliment each other.

**PART 2 - Experiment Tracking with MLFlow**

In [ ]:
!pip install mlflow
import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 74.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.5/857.5 kB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20

In [ ]:
#Create a new MLflow experiment called "olist-satisfaction"
mlflow.set_tracking_uri("mlruns")
mlflow.set_experiment("olist-satisfaction")

print(f"Tracking URI: {mlflow.get_tracking_uri()}")

2026/04/21 19:40:57 INFO mlflow.tracking.fluent: Experiment with name 'olist-satisfaction' does not exist. Creating a new experiment.


Tracking URI: mlruns


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, precision_score, recall_score


In [ ]:
#Log at least 2 model training runs from HW 2 or 3. for each, log parameteres, metrics, artifact
#Log 1
features =  ['delivery_days', 'delivery_vs_estimated', 'freight_value',
            'product_category_name', 'seller_state', 'payment_type_main',
            'seller_historical_average_rating', 'is_new_seller',
            'num_items', 'payment_value_total', 'order_hour', 'order_dayofweek']

X = df_model[features]
y = df_model['is_positive_review']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify = y)

numeric_features = ['delivery_days', 'delivery_vs_estimated', 'freight_value',
                    'seller_historical_average_rating', 'is_new_seller',
                    'num_items', 'payment_value_total', 'order_hour']
categorical_features = ['product_category_name', 'seller_state',
                        'payment_type_main', 'order_dayofweek']

preprocessor = ColumnTransformer([
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]), numeric_features),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ]), categorical_features)
])

# Train Logistic Regression
with mlflow.start_run(run_name="logistic-regression-baseline"):
    lr = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', LogisticRegression(max_iter=1000, random_state=42))
    ])
    lr.fit(X_train, y_train)

    y_pred = lr.predict(X_test)
    y_prob = lr.predict_proba(X_test)[:, 1]

    # Log parameters
    mlflow.log_param("model_type", "LogisticRegression")
    mlflow.log_param("max_iter", 1000)
    mlflow.log_param("random_state", 42)

    # Log metrics
    mlflow.log_metric("accuracy", round(accuracy_score(y_test, y_pred), 4))
    mlflow.log_metric("precision", round(precision_score(y_test, y_pred), 4))
    mlflow.log_metric("recall", round(recall_score(y_test, y_pred), 4))
    mlflow.log_metric("f1", round(f1_score(y_test, y_pred), 4))
    mlflow.log_metric("roc_auc", round(roc_auc_score(y_test, y_prob), 4))


    mlflow.sklearn.log_model(lr, "model")

    print("Run 1 - Logistic Regression")
    print(f"  Accuracy: {round(accuracy_score(y_test, y_pred), 4)}")
    print(f"  Precision: {round(precision_score(y_test, y_pred), 4)}")
    print(f"  Recall: {round(recall_score(y_test, y_pred), 4)}")
    print(f"  F1: {round(f1_score(y_test, y_pred), 4)}")
    print(f"  ROC AUC: {round(roc_auc_score(y_test, y_prob), 4)}")


2026/04/21 19:41:08 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/21 19:41:08 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Run 1 - Logistic Regression
  Accuracy: 0.7965
  Precision: 0.8
  Recall: 0.9811
  F1: 0.8814
  ROC AUC: 0.6933


In [ ]:
#Log2
from lightgbm import LGBMClassifier

with mlflow.start_run(run_name="tuned-lightgbm"):
    lgbm = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', LGBMClassifier(
            n_estimators=100,
            random_state=42,
            verbose=-1
        ))
    ])
    lgbm.fit(X_train, y_train)

    y_pred = lgbm.predict(X_test)
    y_prob = lgbm.predict_proba(X_test)[:, 1]

    # Log parameters
    mlflow.log_param("model_type", "LightGBM")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("random_state", 42)

    # Log metrics
    mlflow.log_metric("accuracy", round(accuracy_score(y_test, y_pred), 4))
    mlflow.log_metric("precision", round(precision_score(y_test, y_pred), 4))
    mlflow.log_metric("recall", round(recall_score(y_test, y_pred), 4))
    mlflow.log_metric("f1", round(f1_score(y_test, y_pred), 4))
    mlflow.log_metric("roc_auc", round(roc_auc_score(y_test, y_prob), 4))

    # Log model artifact
    mlflow.sklearn.log_model(lgbm, "model")

    print("Run 2 - LightGBM")
    print(f"  Accuracy: {round(accuracy_score(y_test, y_pred), 4)}")
    print(f"  Precision: {round(precision_score(y_test, y_pred), 4)}")
    print(f"  Recall: {round(recall_score(y_test, y_pred), 4)}")
    print(f"  F1: {round(f1_score(y_test, y_pred), 4)}")
    print(f"  ROC AUC: {round(roc_auc_score(y_test, y_prob), 4)}")


2026/04/21 19:41:28 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/21 19:41:28 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Run 2 - LightGBM
  Accuracy: 0.8233
  Precision: 0.8273
  Recall: 0.974
  F1: 0.8947
  ROC AUC: 0.7437


In [ ]:
#All logged tuns

client = MlflowClient()
experiment = client.get_experiment_by_name("olist-satisfaction")
runs = client.search_runs(
    experiment_ids= [experiment.experiment_id],
    order_by=["metrics.roc_auc DESC"]
)

print("All logged runs:")
for run in runs:
    print(f"  Run ID: {run.info.run_id}")
    print(f"Model Type: {run.data.params.get('model_type')}")
    print(f"  Accuracy: {run.data.metrics.get('accuracy')}")
    print(f"  Precision: {run.data.metrics.get('precision')}")
    print(f"  Recall: {run.data.metrics.get('recall')}")
    print(f"  F1: {run.data.metrics.get('f1')}")
    print(f"  ROC AUC: {run.data.metrics.get('roc_auc')}")
    print()

All logged runs:
  Run ID: e91ecc7fca6e4d7bbef219ed1dcc35b5
Model Type: LightGBM
  Accuracy: 0.8233
  Precision: 0.8273
  Recall: 0.974
  F1: 0.8947
  ROC AUC: 0.7437

  Run ID: 852f7ee1e7c948a7b27a00db18037e60
Model Type: LogisticRegression
  Accuracy: 0.7965
  Precision: 0.8
  Recall: 0.9811
  F1: 0.8814
  ROC AUC: 0.6933



In [ ]:
# Save mlruns folder to Google Drive
import shutil
shutil.copytree('mlruns', '/content/drive/MyDrive/mlruns')
print("mlruns saved to Google Drive!")

mlruns saved to Google Drive!


In [ ]:
#Register best model
best_run_id = "e91ecc7fca6e4d7bbef219ed1dcc35b5"
model_uri = f"runs:/{best_run_id}/model"
registered_model = mlflow.register_model(model_uri, "olist-satisfaction-model")

print("Model Registered")
print(f"Name: {registered_model.name}")
print(f"Version: {registered_model.version}")
print(f"Run ID: {registered_model.run_id}")

Successfully registered model 'olist-satisfaction-model'.
2026/04/21 20:04:01 WARNING mlflow.tracking._model_registry.fluent: Run with id e91ecc7fca6e4d7bbef219ed1dcc35b5 has no artifacts at artifact path 'model', registering model based on models:/m-b86e1ea1f54b4c9099fc2964d0f2fa42 instead


Model Registered
Name: olist-satisfaction-model
Version: 1
Run ID: e91ecc7fca6e4d7bbef219ed1dcc35b5


Created version '1' of model 'olist-satisfaction-model'.


In [ ]:
#Transition model to "Production" stage
client.transition_model_version_stage(
    name = "olist-satisfaction-model",
    version = 1,
    stage = "Production"
)

print("Model Transitioned to Production")
print("Model: olist-satisfaction-model")
print("Version: 1")
print("Stage: Production")

Model Transitioned to Production
Model: olist-satisfaction-model
Version: 1
Stage: Production


In [ ]:
import shutil
import os
if os.path.exists('/content/drive/MyDrive/mlruns'):
    shutil.rmtree('/content/drive/MyDrive/mlruns')
shutil.copytree('mlruns', '/content/drive/MyDrive/mlruns')
print("mlruns updated in Google Drive!")

mlruns updated in Google Drive!
